Run once before all the other notebooks. \
Inizializes databases, defines global variables, verifies raw file.

# 1. GLOBAL CONFIG
Variables used by all the other notebooks. \
Edit here if path changes or new cities are added.

In [0]:
%run ./00_config.py

# 2. CREA DATABASES
CREATE DATABASE IF NOT EXISTS = idempotent, safe to re-run


In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gtfs_bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS gtfs_silver")
spark.sql("CREATE DATABASE IF NOT EXISTS gtfs_gold")

print("✓ Databases ready: gtfs_bronze, gtfs_silver, gtfs_gold")

# 3. FILE RAW CHECK
Checks that required files exist for each city. \
Prints warning if file are missing - doesn't block execution.

In [0]:
def verify_raw_files():
    for city, folder in CITIES.items():
        path = f"{BASE_PATH}/{folder}"
        
        # file presenti su Volumes
        available = [f.name.replace("/", "") for f in dbutils.fs.ls(path)]
        
        # per Roma saltiamo calendar.txt — è normale che manchi
        expected = [
            t for t in GTFS_TABLES 
            if not (city in CITIES_WITHOUT_CALENDAR and t == "calendar")
            ]
        
        missing = [t for t in expected if f"{t}.txt" not in available]
        
        if missing:
            print(f"  ⚠️  {city}: missing files → {missing}")
        else:
            print(f"  ✓ {city}: all files present")

In [0]:
print("Raw files verification:")
verify_raw_files()